# **Data Balancing Notebook**

## **0. Library import**

This section imports the libraries and modules needed for data balancing. This includes data processing libraries such as pandas, sklearn for data splitting, and custom modules for feature engineering and data balancing.

In [1]:
import os
import sys

# Add the root path into the python path
root_path = os.path.abspath(os.path.join(".."))
if not root_path in sys.path:
    sys.path.insert(0, root_path)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.config import BRFSS_FILTERING_FILE_PATH, BALANCING_STRATEGY, TRAIN_SIZE, N_JOBS
from src.features import DiabetesFeatureEngineering
from src.balancing import OverSamplingBalancer, UnderSamplingBalancer, HybridSamplingBalancer

## **1. Load data**

Load preprocessed data from a CSV file. This data has undergone filtering and cleaning from previous steps, containing information about health indicators and the Diabetes target variable with three classes: No diabetes (0), Pre-diabetes (1), and Diabetes (2).

In [3]:
# Load dataset
df = pd.read_csv(BRFSS_FILTERING_FILE_PATH)
df.head()

,Diabetes,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Year
0,2.0,1.0,1.0,1.0,27.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,1.0,0.0,11.0,6.0,6.0,2017
1,0.0,1.0,0.0,1.0,29.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,1.0,10.0,6.0,8.0,2017
2,0.0,0.0,0.0,1.0,23.0,1.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,14.0,0.0,0.0,10.0,2.0,2.0,2017
3,0.0,1.0,0.0,1.0,27.0,1.0,0.0,1.0,1.0,0.0,...,0.0,3.0,0.0,6.0,0.0,1.0,12.0,4.0,4.0,2017
4,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,0.0,0.0,...,0.0,3.0,0.0,0.0,0.0,1.0,10.0,5.0,8.0,2017


## **2. Feature engineering**

Apply feature engineering techniques to create new features from the original data to improve model performance. This process includes creating composite metrics such as Health Score, Risk Score, and categorical variables such as BMI Category, Age Group. The goal is to create highly informative features that help the machine learning model recognize important patterns in the data.

Generate composite indices from multiple related variables:

- **Health Score**: Composite health index based on GenHlth, MentHlth, PhysHlth
- **Risk Score**: Risk score based on HighBP, HighChol, HeartDiseaseorAttack, Stroke
- **Lifestyle Score**: Lifestyle score based on PhysActivity and negative factors such as HvyAlcoholConsump, Smoker
- **Cardio Risk**: Cardiovascular risk based on HighBP, HighChol and BMI obesity

Convert continuous variables into meaningful categories:

- **BMI Category**: BMI classification according to WHO standards (Underweight, Normal, Pre-obesity, Obesity class I-III)
- **Age Group**: Age grouping into meaningful ranges (Young, Middle-aged, Senior, Elderly)


In [4]:
# Apply feature engineering for the original dataset
diabetes_feature_engineering = DiabetesFeatureEngineering(log_file="../logs/diabetes_feature_engineering_pipeline.log")
processed_df, _, _ = diabetes_feature_engineering.process_all(df=df)
processed_df.shape

2025-08-17 01:33:09,229 - [src.features] - INFO - DiabetesFeatureEngineering initialized successfully
2025-08-17 01:33:09,229 - [src.features] - INFO - ============================================================
2025-08-17 01:33:09,229 - [src.features] - INFO - STARTING COMPLETE FEATURE ENGINEERING PIPELINE
2025-08-17 01:33:09,230 - [src.features] - INFO - ============================================================
2025-08-17 01:33:09,230 - [src.features] - INFO - Initial dataset shape: (787602, 21)
2025-08-17 01:33:09,231 - [src.features] - INFO - Starting null values removal process...
2025-08-17 01:33:09,246 - [src.features] - INFO - No null values found in the dataset
2025-08-17 01:33:09,312 - [src.features] - INFO - Null values removal completed. Removed 0 rows (0.00%)
2025-08-17 01:33:09,313 - [src.features] - INFO - Dataset shape: 787602 -> 787602 rows
2025-08-17 01:33:09,314 - [src.features] - INFO - After null removal: (787602, 21)
2025-08-17 01:33:09,314 - [src.features] - 

(702516, 15)

In [5]:
# Check the number of samples on each class
processed_df["Diabetes"].value_counts()

Diabetes
0.0    574442
2.0    111184
1.0     16890
Name: count, dtype: int64

In [6]:
# Get features and target
X = processed_df.drop(columns=["Diabetes"])
y = processed_df["Diabetes"]

In [7]:
# Split training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    train_size=TRAIN_SIZE, 
    random_state=42, 
    stratify=y
)

## **3. Data balancing**

Addressing the data imbalance issue in the BRFSS dataset, where the "No diabetes" class is the majority (459,553 samples) compared to "Pre-diabetes" (13,512 samples) and "Diabetes" (88,947 samples). Data balance is extremely important to ensure that the model is not biased towards the majority class and can predict accurately for all classes.

### **3.1 Over sampling**

The method increases the number of samples of the minority class by generating new data to balance the majority class.

In [8]:
# Initialize OverSamplingBalancer
over_sampling_balancer = OverSamplingBalancer(log_file="../logs/over_sampling_balancer.log")

2025-08-17 01:33:13,867 - [src.balancing] - INFO - OverSamplingBalancer initialized successfully


#### **3.1.1 Random Over Sampling**

The simplest technique in over-sampling, creating additional samples for the minority class by randomly copying existing samples. This method is easy to implement but can lead to overfitting due to completely repeating old samples. Results after applying: Class 0: 574,477, Class 1: 150,000, Class 2: 100,000 samples.

In [9]:
# Apply RandomOverSampling on training dataset
ros_X_train, ros_y_train = over_sampling_balancer.apply_random_oversampling(
    X=X_train, 
    y=y_train, 
    sampling_strategy=BALANCING_STRATEGY
)

2025-08-17 01:33:13,919 - [src.balancing] - INFO - Starting Random Over Sampling process...
2025-08-17 01:33:13,928 - [src.balancing] - INFO - Original class distribution:
2025-08-17 01:33:13,929 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-08-17 01:33:13,930 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-08-17 01:33:13,930 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-08-17 01:33:14,097 - [src.balancing] - INFO - New class distribution after Random Over Sampling:
2025-08-17 01:33:14,098 - [src.balancing] - INFO -   - Class 0.0: 574477 samples (+114924)
2025-08-17 01:33:14,099 - [src.balancing] - INFO -   - Class 1.0: 150000 samples (+136488)
2025-08-17 01:33:14,099 - [src.balancing] - INFO -   - Class 2.0: 100000 samples (+11053)
2025-08-17 01:33:14,100 - [src.balancing] - INFO - Dataset size: 562012 -> 824477 samples (+262465)
2025-08-17 01:33:14,101 - [src.balancing] - INFO - Random Over Sampling completed successfully


In [10]:
# Count the number of samples on each class after balancing
ros_y_train.value_counts()

Diabetes
0.0    574477
1.0    150000
2.0    100000
Name: count, dtype: int64

#### **3.1.2 SMOTE**

Synthetic Minority Oversampling Technique—a more sophisticated method that generates new synthetic samples for the minority class by interpolating between existing samples and their nearest neighbors. SMOTE helps generate more diverse data, reducing the risk of overfitting compared to random oversampling. The result is similar to Random Over Sampling with the creation of higher quality synthetic samples.

In [11]:
# Apply SMOTE on the training set
smote_X_train, smote_y_train = over_sampling_balancer.apply_smote(
    X=X_train, 
    y=y_train, 
    sampling_strategy=BALANCING_STRATEGY
)

2025-08-17 01:33:14,321 - [src.balancing] - INFO - Starting SMOTE process...
2025-08-17 01:33:14,327 - [src.balancing] - INFO - Original class distribution:
2025-08-17 01:33:14,328 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-08-17 01:33:14,329 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-08-17 01:33:14,329 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-08-17 01:34:30,899 - [src.balancing] - INFO - New class distribution after SMOTE:
2025-08-17 01:34:30,899 - [src.balancing] - INFO -   - Class 0.0: 574477 samples (+114924 synthetic)
2025-08-17 01:34:30,899 - [src.balancing] - INFO -   - Class 1.0: 150000 samples (+136488 synthetic)
2025-08-17 01:34:30,900 - [src.balancing] - INFO -   - Class 2.0: 100000 samples (+11053 synthetic)
2025-08-17 01:34:30,900 - [src.balancing] - INFO - Dataset size: 562012 -> 824477 samples (+262465 synthetic)
2025-08-17 01:34:30,900 - [src.balancing] - INFO - SMOTE completed successfully


In [12]:
# Count the number of samples on each class after balancing
smote_y_train.value_counts()

Diabetes
0.0    574477
1.0    150000
2.0    100000
Name: count, dtype: int64

### **3.2 Under Sampling**

The method reduces the number of samples of the majority class to balance with the minority classes, which reduces training time but may lose important information.

In [13]:
# Initialize UnderSamplingBalancer
under_sampling_balancer = UnderSamplingBalancer(log_file="../logs/under_sampling_balancer.log")

2025-08-17 01:34:31,099 - [src.balancing] - INFO - UnderSamplingBalancer initialized successfully


#### **3.2.1 Random Under Sampling**

Randomly remove samples from the majority class to achieve balance. This method is simple and fast but risks losing important information when removing valuable samples. Result: each class has 13,512 samples left, for a total of 40,536 samples.

In [14]:
# Apply RandomUnderSampling on the training set
rus_X_train, rus_y_train = under_sampling_balancer.apply_random_undersampling(
    X=X_train, 
    y=y_train
)

2025-08-17 01:34:31,215 - [src.balancing] - INFO - Starting Random Under Sampling process...
2025-08-17 01:34:31,224 - [src.balancing] - INFO - Original class distribution:
2025-08-17 01:34:31,225 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-08-17 01:34:31,226 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-08-17 01:34:31,227 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-08-17 01:34:31,336 - [src.balancing] - INFO - New class distribution after Random Under Sampling:
2025-08-17 01:34:31,337 - [src.balancing] - INFO -   - Class 0.0: 13512 samples (-446041)
2025-08-17 01:34:31,338 - [src.balancing] - INFO -   - Class 1.0: 13512 samples (-0)
2025-08-17 01:34:31,339 - [src.balancing] - INFO -   - Class 2.0: 13512 samples (-75435)
2025-08-17 01:34:31,340 - [src.balancing] - INFO - Dataset size: 562012 -> 40536 samples (-521476)
2025-08-17 01:34:31,341 - [src.balancing] - INFO - Random Under Sampling completed successfully


In [15]:
# Count the number of samples on each class after balancing
rus_y_train.value_counts()

Diabetes
0.0    13512
1.0    13512
2.0    13512
Name: count, dtype: int64

#### **3.2.2 TomekLinks**

The under-sampling method is smarter, only discarding the majority class samples that are close to the boundary decision and may cause noise. TomekLinks identifies pairs of samples from different classes that are nearest neighbors to each other and discards the majority class sample in that pair. Results: 444,228 (Class 0), 13,512 (Class 1), 74,901 (Class 2)—retains more information than random undersampling

In [16]:
# Apply TomekLinks on the training set
tomek_X_train, tomek_y_train = under_sampling_balancer.apply_tomek_links(
    X=X_train, 
    y=y_train, 
    n_jobs=N_JOBS
)

2025-08-17 01:34:31,631 - [src.balancing] - INFO - Starting Tomek Links process...
2025-08-17 01:34:31,643 - [src.balancing] - INFO - Original class distribution:
2025-08-17 01:34:31,644 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-08-17 01:34:31,645 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-08-17 01:34:31,646 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-08-17 01:34:59,073 - [src.balancing] - INFO - New class distribution after Tomek Links:
2025-08-17 01:34:59,074 - [src.balancing] - INFO -   - Class 0.0: 444228 samples (-15325 Tomek links)
2025-08-17 01:34:59,074 - [src.balancing] - INFO -   - Class 1.0: 13512 samples (unchanged)
2025-08-17 01:34:59,075 - [src.balancing] - INFO -   - Class 2.0: 74901 samples (-14046 Tomek links)
2025-08-17 01:34:59,075 - [src.balancing] - INFO - Dataset size: 562012 -> 532641 samples (-29371 Tomek links)
2025-08-17 01:34:59,076 - [src.balancing] - INFO - Tomek Links processing completed successfully


In [17]:
# Count the number of samples on each class after balancing
tomek_y_train.value_counts()

Diabetes
0.0    444228
2.0     74901
1.0     13512
Name: count, dtype: int64

### **3.3 Hyprid**

Combine both over-sampling and under-sampling to get the best of both worlds, creating a balanced dataset with the highest quality.

In [18]:
# Initialize HybridSamplingBalancer
hyprid_sampler = HybridSamplingBalancer(log_file="../logs/hybrid_sampler.log")

2025-08-17 01:34:59,309 - [src.balancing] - INFO - HybridSamplingBalancer initialized successfully


#### **3.3.1 SMOTETomek**

Combining SMOTE and TomekLinks: first apply SMOTE to upsample the minority class, then use TomekLinks to clean the boundary and remove noisy samples. This method produces a balanced dataset with high quality, reduces noise, improves classification ability at the boundary. Results: 564,423 (Class 0), 149,036 (Class 1), 90,450 (Class 2).


In [19]:
# Apply SMOTETomek on the training set
smote_tomek_X_train, smote_tomek_y_train = hyprid_sampler.apply_smote_tomek(
    X=X_train, 
    y=y_train, 
    sampling_strategy=BALANCING_STRATEGY, 
    n_jobs=N_JOBS
)

2025-08-17 01:34:59,431 - [src.balancing] - INFO - Starting SMOTETomek hybrid sampling process...
2025-08-17 01:34:59,439 - [src.balancing] - INFO - Original class distribution:
2025-08-17 01:34:59,440 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-08-17 01:34:59,441 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-08-17 01:34:59,442 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-08-17 01:37:06,911 - [src.balancing] - INFO - New class distribution after SMOTETomek:
2025-08-17 01:37:06,911 - [src.balancing] - INFO -   - Class 0.0: 564423 samples (+104870 net)
2025-08-17 01:37:06,912 - [src.balancing] - INFO -   - Class 1.0: 149036 samples (+135524 net)
2025-08-17 01:37:06,912 - [src.balancing] - INFO -   - Class 2.0: 90450 samples (+1503 net)
2025-08-17 01:37:06,912 - [src.balancing] - INFO - Dataset size: 562012 -> 803909 samples (+241897)
2025-08-17 01:37:06,913 - [src.balancing] - INFO - SMOTETomek processing completed successfully


In [20]:
# Count the number of samples on each class after balancing
smote_tomek_y_train.value_counts()

Diabetes
0.0    564423
1.0    149036
2.0     90450
Name: count, dtype: int64

#### **3.3.2 SMOTEENN**

Combining SMOTE with Edited Nearest Neighbors (ENN): after SMOTE generates the synthetic sample, ENN removes the samples that are misclassified by the majority of k-nearest neighbors. This method is more aggressive in cleaning the data, may remove more samples than necessary but creates a clearer decision boundary. Results: 405,203 (Class 0), 106,148 (Class 1), 8,988 (Class 2).

In [21]:
# Apply SMOTEENN on training set
smoteen_X_train, smoteen_y_train = hyprid_sampler.apply_smote_enn(
    X=X_train, 
    y=y_train, 
    sampling_strategy=BALANCING_STRATEGY, 
    n_jobs=N_JOBS
)

2025-08-17 01:37:07,127 - [src.balancing] - INFO - Starting SMOTEENN hybrid sampling process...
2025-08-17 01:37:07,135 - [src.balancing] - INFO - Original class distribution:
2025-08-17 01:37:07,136 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-08-17 01:37:07,137 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-08-17 01:37:07,138 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-08-17 01:39:14,372 - [src.balancing] - INFO - New class distribution after SMOTEENN:
2025-08-17 01:39:14,373 - [src.balancing] - INFO -   - Class 0.0: 405203 samples (-54350 net)
2025-08-17 01:39:14,374 - [src.balancing] - INFO -   - Class 1.0: 106148 samples (+92636 net)
2025-08-17 01:39:14,374 - [src.balancing] - INFO -   - Class 2.0: 8988 samples (-79959 net)
2025-08-17 01:39:14,374 - [src.balancing] - INFO - Dataset size: 562012 -> 520339 samples (-41673)
2025-08-17 01:39:14,376 - [src.balancing] - INFO - SMOTEENN processing completed successfully


In [22]:
# Count the number of samples on each class after balancing
smoteen_y_train.value_counts()

Diabetes
0.0    405203
1.0    106148
2.0      8988
Name: count, dtype: int64

### **3.4 Comparison of results of methods**

The summary table shows the clear differences between the methods:
- **Over-sampling methods** produce the largest dataset, suitable when all information needs to be kept
- **Under-sampling methods** produce the smallest dataset, suitable when computational resources are limited
- **Hybrid methods** balance size and quality, often giving the best results in practice

The choice of the appropriate method depends on the specific characteristics of the data, the available computational resources, and the model performance requirements.

| Method             | Class 0     | Class 1     | Class 2    | Total Samples  |
|:-------------------|:------------|:------------|:-----------|:---------------|
| Original           | 459,553     | 13,512      | 88,947     | 562,012        |
| Random Oversample  | 574,477     | 150,000     | 100,000    | 824,477        |
| SMOTE              | 574,477     | 150,000     | 100,000    | 824,477        |
| Random Undersample | 13,512      | 13,512      | 13,512     | 40,536         |
| TomekLinks         | 444,228     | 13,512      | 74,901     | 532,641        |
| SMOTETomek         | 564,423     | 149,036     | 90,450     | 803,909        |
| SMOTEEN            | 405,203     | 106,148     | 8,988      | 520,339        |